# poblar en Colab

Calcula la población expuesta a la degradación de tierras a partir de un censo por radio, una cobertura del suelo y un mapa de degradación. Los criterios están en el archivo de método; los datos de cada ejecución se indican acá, archivo por archivo, con su ruta (el nombre de los archivos no importa).

1. Celda 3: año, método y la ruta de cada insumo. Solo hacen falta los insumos de las etapas que se quieran correr.
2. Ejecutar las celdas en orden y revisar los controles (celda 5).
3. Con dos años ejecutados, unir las tablas (celda 7).

Los rasters se procesan en el disco local de Colab y lo terminado se copia a Drive a medida que avanza. Si la sesión se corta, volver a ejecutar: con `ETAPAS = 'todas'` se corren las etapas que tienen sus insumos; para seguir desde donde quedó, indicar solo las que faltan.


In [ ]:
# 1. Instalación
# Se clona el repositorio y se instala desde el clon: el código y las configuraciones
# son de la misma versión, y la ejecución registra el commit en entorno.txt.
# Para reproducir una versión exacta, indicar una etiqueta o un commit en VERSION.
VERSION = 'main'
!rm -rf /content/poblar_repo
!git clone -q https://github.com/AugustoBellanich/poblar /content/poblar_repo
!git -C /content/poblar_repo checkout -q {VERSION}
!pip install -q -e /content/poblar_repo
# La instalación editable se activa recién al reiniciar el intérprete; con esto
# la librería se puede importar en la misma sesión, sin reiniciar.
import sys, importlib
if '/content/poblar_repo' not in sys.path:
    sys.path.insert(0, '/content/poblar_repo')
importlib.invalidate_caches()
# Si en esta sesión ya se había importado una versión anterior, se descarta para usar la nueva.
for _m in [m for m in sys.modules if m == 'poblar' or m.startswith('poblar.')]:
    del sys.modules[_m]
!git -C /content/poblar_repo log -1 --format='Versión: %h — %s (%cs)'

# Herramientas de GDAL (gdalwarp, gdal_rasterize) para las etapas 2 y 3
!which gdalwarp > /dev/null || (apt-get -qq update && apt-get -qq install -y gdal-bin > /dev/null)
!gdalinfo --version


In [ ]:
# 2. Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Parámetros de la ejecución
from pathlib import Path

ANIO    = 2022
METODO  = Path('/content/poblar_repo/aplicaciones/prais2026/metodo.yaml')
# Datos oficiales del período (huellas de los insumos y cifras publicadas). Opcional: None si se usan otros datos.
REGISTRO = Path(f'/content/poblar_repo/aplicaciones/prais2026/periodos/{ANIO}.yaml')

# Ruta de cada insumo, con el nombre que tenga. Dejar en '' los que no se usan.
D = '/content/drive/MyDrive/PRAIS/poblar/insumos'
INSUMOS = {
    'tabla_censo': f'{D}/tabla_censo{ANIO}.xlsx',                 # tabla censal por radio
    'radios':      f'{D}/radios_{ANIO}.zip',                      # cartografía de radios
    'regiones':    f'{D}/regiones_PRAIS.gpkg',                    # regiones de reporte
    'cobertura':   f'{D}/MB_prais_{2015 if ANIO == 2010 else 2023}_100m.tif',   # cobertura del suelo
    'degradacion': f'{D}/degradacion_{"referencia" if ANIO == 2010 else "reporte"}_{ANIO}.tif',  # mapa de degradación
}
SALIDA  = Path('/content/drive/MyDrive/PRAIS/poblar/ejecuciones')    # una subcarpeta por ejecución
TRABAJO = Path('/content/trabajo')                                   # disco local de Colab
ETAPAS  = 'todas'      # o una lista, por ejemplo ['cruce', 'tablas']

# Verificación previa: que existan los archivos indicados
INSUMOS = {rol: ruta for rol, ruta in INSUMOS.items() if ruta}
faltan = [f'{rol}: {ruta}' for rol, ruta in INSUMOS.items() if not Path(ruta).exists()]
if faltan:
    raise FileNotFoundError('No se encontraron estos insumos:\n  ' + '\n  '.join(faltan))
print('Insumos:', ', '.join(INSUMOS))


In [ ]:
# 4. Ejecución
# Si un control bloqueante falla, la ejecución se detiene y se informa el motivo;
# los controles y el balance quedan igual en la carpeta para revisarlos.
from poblar import ejecutar
from poblar.trazabilidad import ControlFallido

try:
    ejecucion = ejecutar(METODO, anio=ANIO, insumos=INSUMOS, registro=REGISTRO,
                         salida=SALIDA, etapas=ETAPAS, trabajo=TRABAJO)
    CARPETA = ejecucion.carpeta
    print('\nEjecución completa:', CARPETA)
except ControlFallido as e:
    CARPETA = e.carpeta
    print('\nLa ejecución se detuvo:', e)
    print('Carpeta:', CARPETA)


In [ ]:
# 5. Controles y balance
import pandas as pd
pd.set_option('display.max_colwidth', 120)
controles = pd.read_csv(CARPETA / 'controles.csv')
print(f"Controles: {len(controles)} | sin cumplir: {(~controles['ok']).sum()}")
display(controles[['etapa', 'nombre', 'ok', 'detalle', 'bloqueante']])
display(pd.read_csv(CARPETA / 'balance.csv'))

## Resultados

Población expuesta (nacional y por región) y rasters publicados de la ejecución.


In [ ]:
# 6. Resultados de la ejecución
for archivo in ['4_cruce/resumen_expuesta.csv', '5_publicacion/publicacion.csv']:
    ruta = CARPETA / archivo
    if ruta.exists():
        print(archivo); display(pd.read_csv(ruta))


## Tablas de dos períodos

Con la etapa `tablas` ejecutada en los dos años, une las tablas con una columna por período.


In [ ]:
# 7. Tablas de dos períodos
from poblar.tablas import unir_periodos

ANIOS = (2010, 2022)
BASE = CARPETA.name.rsplit('_', 1)[0]                 # por ejemplo 'prais2026'
EJEC = {a: SALIDA / f'{BASE}_{a}' for a in ANIOS}
DEST = SALIDA / f'{BASE}_tablas'
for nombre, t in unir_periodos(EJEC, DEST).items():
    print(nombre); display(t)
print('Tablas en', DEST)
